# Hand Landmark Gesture Recorder ⏺️
Este notebook grava as posições dos 21 landmarks da mão via webcam e salva em um arquivo CSV. 

**Nota:** Corrigido o erro de importação do `mediapipe.solutions` usando lógica estável.

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import csv
import os
import time

## Configurações do Gravador
Defina o nome do gesto que você vai gravar e o nome do arquivo.

In [ ]:
LABEL_NAME = "thumbs_up"  # Altere para o gesto que quer gravar (ex: "circle", "stop")
CSV_PATH = "../storage/csv/hand_landmarks.csv"
MODEL_PATH = '../storage/nn_models/gesture_recognizer.task'

# Se o arquivo não existir, criamos o cabeçalho
if not os.path.exists(os.path.dirname(CSV_PATH)):
    os.makedirs(os.path.dirname(CSV_PATH))

header = ['label']
for i in range(21):
    header.extend([f'x{i}', f'y{i}', f'z{i}'])

if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
    print(f"Arquivo criado: {CSV_PATH}")

## Lógica de Gravação e Visualização
Usaremos o detector de gestos para extrair os pontos.

In [ ]:
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7
)
recognizer = vision.GestureRecognizer.create_from_options(options)

def save_landmarks_to_csv(landmarks, label):
    row = [label]
    for lm in landmarks:
        row.extend([lm.x, lm.y, lm.z])
    
    with open(CSV_PATH, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(row)

def draw_manual_landmarks(image, hand_landmarks):
    h, w, _ = image.shape
    
    # Definindo conexões da mão
    CONNECTIONS = [
        (0,1), (1,2), (2,3), (3,4),       # Polegar
        (0,5), (5,6), (6,7), (7,8),       # Indicador
        (5,9), (9,10), (10,11), (11,12),  # Médio
        (9,13), (13,14), (14,15), (15,16),# Anelar
        (13,17), (17,18), (18,19), (19,20), (0,17) # Mínimo e base
    ]

    # 1. Desenhar conexões
    for start_idx, end_idx in CONNECTIONS:
        pt1 = (int(hand_landmarks[start_idx].x * w), int(hand_landmarks[start_idx].y * h))
        pt2 = (int(hand_landmarks[end_idx].x * w), int(hand_landmarks[end_idx].y * h))
        cv2.line(image, pt1, pt2, (255, 255, 255), 1)

    # 2. Desenhar pontos
    for landmark in hand_landmarks:
        cx, cy = int(landmark.x * w), int(landmark.y * h)
        cv2.circle(image, (cx, cy), 3, (0, 255, 0), -1)
            
    return image

## Captura com Gravação Automática
Pressione **'q'** para finalizar.

In [ ]:
cap = cv2.VideoCapture(0)
recorded_count = 0

print(f"Gravando gestos para a label: {LABEL_NAME}")
print("Pressione 'q' para parar.")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    timestamp = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)
    result = recognizer.recognize_for_video(mp_image, timestamp)

    if result.hand_landmarks:
        landmarks = result.hand_landmarks[0]
        save_landmarks_to_csv(landmarks, LABEL_NAME)
        recorded_count += 1
        
        # Feedback visual
        frame = draw_manual_landmarks(frame, landmarks)
        cv2.putText(frame, f"Records: {recorded_count}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow('Gesture Data Recorder', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Finalizado! {recorded_count} amostras salvas em {CSV_PATH}.")